# Bronze — Lossless Raw Ingestion

Reads `orders.csv` and `transactions.csv` from a configurable UC Volume and writes them to managed Delta tables with:

- **Lossless STRING-only schema** — every source column read as `STRING` via an explicit schema (no inference); malformed records are preserved, never rejected.
- **Audit metadata** — `_ingested_at`, `_source_file`, `_batch_id`, and `_row_hash` (SHA-256 fingerprint) appended to every row.
- **Idempotent loading** — existing rows for the same `_source_file` are deleted before appending, so re-running the notebook produces the same row counts.
- **Reconciliation checks** — Delta row counts verified against source CSV counts; audit columns validated as present and non-null.

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "ecom_poc", "Schema")
dbutils.widgets.text("volume", "raw_files", "Volume")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
VOLUME = dbutils.widgets.get("volume").strip()
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Ensure the target schema exists before any writes
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
print(f"Source volume : {VOLUME_PATH}")
print(f"Target schema : {CATALOG}.{SCHEMA}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone

# Deterministic batch ID — same for all rows in this run, unique across runs
BATCH_ID = f"bronze_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

# Explicit all-STRING schemas — no type inference, raw fidelity preserved
INGEST_SPECS = [
    ("bronze_orders", "orders.csv", StructType([
        StructField("order_id",      StringType(), True),
        StructField("customer_id",    StringType(), True),
        StructField("order_date",     StringType(), True),
        StructField("category",       StringType(), True),
        StructField("quantity",       StringType(), True),
        StructField("unit_price",     StringType(), True),
        StructField("order_status",   StringType(), True),
        StructField("country",        StringType(), True),
    ])),
    ("bronze_transactions", "transactions.csv", StructType([
        StructField("txn_id",         StringType(), True),
        StructField("order_id",       StringType(), True),
        StructField("payment_method", StringType(), True),
        StructField("amount",         StringType(), True),
        StructField("currency",       StringType(), True),
        StructField("txn_status",     StringType(), True),
        StructField("txn_ts",         StringType(), True),
    ])),
]

for table_name, file_name, schema in INGEST_SPECS:
    fqn = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
    src_path = f"{VOLUME_PATH}/{file_name}"
    source_cols = [f.name for f in schema.fields]

    # 1. Lossless read — explicit all-STRING schema, header=true
    df = (spark.read
          .option("header", "true")
          .schema(schema)
          .csv(src_path))

    # 2. Append audit metadata columns (no source column is touched)
    df = (df
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.lit(file_name))
          .withColumn("_batch_id",    F.lit(BATCH_ID))
          .withColumn("_row_hash",   F.sha2(
              F.concat_ws("\u0001", *[F.coalesce(F.col(c), F.lit("\u0000")) for c in source_cols]),
              256
          )))

    # 3. Idempotent write — delete prior rows for this source file, then append
    if spark.catalog.tableExists(fqn):
        spark.sql(f"DELETE FROM {fqn} WHERE _source_file = '{file_name}'")
        df.write.mode("append").saveAsTable(fqn)
        print(f"\u2713 {table_name}: refreshed (deleted old rows for {file_name}, appended fresh)")
    else:
        df.write.mode("overwrite").saveAsTable(fqn)
        print(f"\u2713 {table_name}: created (initial load from {file_name})")

    print(f"  batch_id = {BATCH_ID}")
    print(f"  source   = {src_path}")

In [0]:
from pyspark.sql import functions as F

EXPECTED   = {"bronze_orders": 1_000_000, "bronze_transactions": 1_000_000}
FILES      = {"bronze_orders": "orders.csv", "bronze_transactions": "transactions.csv"}
AUDIT_COLS = ["_ingested_at", "_source_file", "_batch_id", "_row_hash"]

all_passed = True
print("=" * 60)
print("BRONZE RECONCILIATION CHECKS")
print("=" * 60)

for table_name, file_name in FILES.items():
    fqn = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
    src_path = f"{VOLUME_PATH}/{file_name}"

    print(f"\n--- {table_name} ---")

    # 1. Source CSV row count (header=true excludes the header line)
    csv_count = spark.read.option("header", "true").option("inferSchema", "false").csv(src_path).count()

    # 2. Delta table row count
    delta_count = spark.table(fqn).count()

    # 3. Audit column presence
    table_cols = spark.table(fqn).columns
    missing_audit = [c for c in AUDIT_COLS if c not in table_cols]
    cols_ok = len(missing_audit) == 0

    # 4. Audit column non-null check (only if columns exist)
    if cols_ok:
        null_counts = (spark.table(fqn)
            .agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in AUDIT_COLS])
            .collect()[0])
        null_map = {c: null_counts[c] for c in AUDIT_COLS}
        nulls_ok = all(v == 0 for v in null_map.values())
    else:
        null_map = {c: "col missing" for c in AUDIT_COLS}
        nulls_ok = False

    # 5. Evaluate checks
    count_ok = (csv_count == delta_count == EXPECTED[table_name])
    table_pass = count_ok and cols_ok and nulls_ok
    all_passed = all_passed and table_pass

    print(f"  Expected rows:       {EXPECTED[table_name]:,}")
    print(f"  Source CSV rows:     {csv_count:,}")
    print(f"  Delta table rows:    {delta_count:,}")
    print(f"  Row count match:     {'\u2713 PASS' if count_ok else '\u2717 FAIL'}")
    print(f"  Audit cols present:  {'\u2713 PASS' if cols_ok else '\u2717 FAIL \u2014 missing: ' + str(missing_audit)}")
    print(f"  Audit cols non-null: {'\u2713 PASS' if nulls_ok else '\u2717 FAIL \u2014 nulls: ' + str(null_map)}")
    print(f"  OVERALL:             {'\u2713 PASS' if table_pass else '\u2717 FAIL'}")

print(f"\n{'=' * 60}")
print(f"RESULT: {'\u2713 ALL CHECKS PASSED' if all_passed else '\u2717 FAILURES DETECTED'}")
print("=" * 60)

assert all_passed, "Reconciliation FAILED \u2014 see details above."